In [1]:
import csv
import io
import re
import time
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
def show_iteration_table(result, decimals=8):
    """
    Hiển thị bảng:
        Iteration | x^k (vector) | Residual | Final

    Residual:
        r_k = ||x^k - x^(k-1)||_2
    """

    rows = []

    for k, x in sorted(result["res"].items()):

        vector_str = np.array2string(
            x,
            precision=decimals,
            separator=", ",
            suppress_small=False,
            threshold=np.inf,
            max_line_width=100000
        )

        rows.append({
            "Iteration (k)": k,
            "x^k (vector)": vector_str,
            "Residual ||x^k - x^(k-1)||₂":
                result["residual_history"].get(
                    k, np.nan
                ),
            "Final": k == result["iterations"]
        })

    table = pd.DataFrame(rows)

    table["Residual ||x^k - x^(k-1)||₂"] = (
        table["Residual ||x^k - x^(k-1)||₂"]
        .map(
            lambda v: f"{v:.3e}"
            if pd.notna(v) else "—"
        )
    )

    display(table)

    return table


### Bài toán bao hàm biến phân
tìm điểm $x$ thỏa mãn:

$$ 0 \in Ax + Bx $$



Với bài toán

$$
\min_{x\in C}\frac12\|Rx-b\|_2^2,
$$

chọn

$$
A=N_C,\qquad B(x)=R^\top(Rx-b).
$$

Khi đó thuật toán trở thành

$$
\boxed{
x^{k+1}
=P_C\bigl(x^k-\gamma R^\top(Rx^k-b)\bigr).
}
$$

Trong đó $P_C$ là phép chiếu mêtric lên tập $C$.
# THUẬT TOÁN FORWARD-BACKWARD SPLITTING METHOD (FBM)

Điều kiện FBM:

- $A$ là toán tử đa trị đơn điệu cực đại (tồn tại resolvent)
- $B$ là toán tử đơn trị thường được giả thiết $\beta$ cocoercive
- $J_{\gamma A} = (I + \gamma A)^{-1}$ là toán tử giải của A
- $\gamma > 0$ là cỡ bước ($0<\gamma<2\beta$)

# Ý TƯỞNG THUẬT TOÁN

Với $\gamma>0$, ta có:

$$
\begin{aligned}
0\in Ax^\ast+Bx^\ast
&\iff -Bx^\ast\in Ax^\ast\\
&\iff -\gamma Bx^\ast\in\gamma Ax^\ast\\
&\iff x^\ast-\gamma Bx^\ast
\in x^\ast+\gamma Ax^\ast\\
&\iff x^\ast-\gamma Bx^\ast
\in (I+\gamma A)x^\ast\\
&\iff x^\ast\in (I+\gamma A)^{-1}
(x^\ast-\gamma Bx^\ast)\\
&\iff \boxed{x^\ast=J_{\gamma A}
(x^\ast-\gamma Bx^\ast)}.
\end{aligned}
$$

Trong đó:
$$
J_{\gamma A}=(I+\gamma A)^{-1}.
$$

trong đó $J_{\gamma A}=(I+\gamma A)^{-1}$.


Suy ra $x^\ast\in\operatorname{Fix}(T)$, với

$$
T=J_{\gamma A}\circ(I-\gamma B).
$$

Do đó, thuật toán lặp được xác định bởi:

$$
\boxed{
x^{k+1}=T(x^k)
=J_{\gamma A}(x^k-\gamma Bx^k),
\quad k\ge 0.
}
$$


Thuật toán FBM tách việc cập nhật tại mỗi bước lặp k thành hai bước nối tiếp:

## 1. Bước Forward:

   Từ điểm hiện tại $x_k$, thực hiện một bước cập nhật trực tiếp theo toán tử B:

   $$
   y_k = x_k - \gamma B(x_k)
   $$

## 2. Bước Backward:

   Từ điểm trung gian $y_k$, áp dụng toán tử toán tử giải của A để thu được điểm lặp tiếp theo:

   $$
   x_{k+1} = J_{\gamma A}(y_k) <=> x_{k+1}
   =
   J_{\gamma A}\left(x_k-\gamma B(x_k)\right)
   $$

Trong trường hợp A = $\partial g$, với g là một hàm lồi, resolvent của $A$ chính là toán tử proximal:

$$
J_{\gamma A}
=
J_{\gamma \partial g}
=
\operatorname{prox}_{\gamma g}
$$

Khi đó bước Backward có thể viết dưới dạng:

$$
x_{k+1}
=
\operatorname{prox}_{\gamma g}(y_k)
$$

Quá trình lặp được thực hiện cho đến khi sai khác giữa hai điểm liên tiếp đủ nhỏ, chẳng hạn:

$$
\|x_{k+1}-x_k\| < \mathrm{tol}
$$

với $\mathrm{tol} > 0$ là ngưỡng sai số cho trước.


### Mã giả Forward–Backward

**Bước 0 (Khởi tạo).**

Chọn $x^0\in\mathcal H$, $\gamma\in(0,2\beta)$,
$\varepsilon>0$, số vòng lặp tối đa $K_{\max}$.
Đặt $k=0$.

**Bước 1.** Tính điểm trung gian:

$$
y^k=x^k-\gamma B(x^k).
$$

**Bước 2.** Tính:

$$
x^{k+1}=J_{\gamma A}(y^k).
$$

**Bước 3.** Nếu

$$
\|x^{k+1}-x^k\|\leq\varepsilon
$$

hoặc $k+1=K_{\max}$, dừng thuật toán.

**Bước 4.** Đặt $k:=k+1$ và quay lại Bước 1.

In [3]:
from FBM import fbm

# Ví dụ
a = np.array([3.0, 4.0])
x0 = np.array([100.0, -100.0])

# Toán tử B
def B(x):
    return x - a

# Phép chiếu lên quả cầu đóng
def proj_ball(x, center, radius):
    d = x - center
    nd = np.linalg.norm(d)

    if nd <= radius:
        return x.copy()

    return center + radius * d / nd

# Toán tử giải
def prox(y, gamma):
    return proj_ball(
        y,
        center=np.zeros(2),
        radius=1.0
    )

# Hằng số Lipschitz
L = 1.0
gamma = 1.0 / L

result = fbm(
    x0=x0,
    B=B,
    prox=prox,
    gamma=gamma,
    max_iter=10000,
    tol=1e-8,
    iter_list=[1, 10, 100, 1000, 10000]
)

table = show_iteration_table(result)

,Iteration (k),x^k (vector),Residual ||x^k - x^(k-1)||₂,Final
0,0,"[ 100., -100.]",—,False
1,1,"[0.6, 0.8]",1.416e+02,False
2,2,"[0.6, 0.8]",0.000e+00,True


# THUẬT TOÁN MODIFIED FORWARD–BACKWARD SPLITTING METHOD (MFBSM) CỦA TSENG

Điều kiện MFBSM:

- $A$ là toán tử đa trị đơn điệu cực đại (tồn tại resolvent).
- $B$ là toán tử đơn trị, đơn điệu và liên tục Lipschitz với hằng số $L>0$
- $J_{\gamma A}=(I+\gamma A)^{-1}$ là toán tử giải của $A$.
- $\gamma>0$ là cỡ bước ($0<\gamma<\frac1L$)

Khác với FBM thông thường, MFBSM không yêu cầu toán tử $B$ phải cocoercive mà chỉ cần đơn điệu và liên tục Lipschitz.

# Ý TƯỞNG THUẬT TOÁN

Tương tự FBM, ta xác định điểm trung gian:

$$
y^k=T(x^k)
=J_{\gamma A}(x^k-\gamma Bx^k).
$$

Tuy nhiên, thay vì lấy trực tiếp $x^{k+1}=y^k$ như FBM, phương pháp MFBSM của Tseng bổ sung thêm một bước hiệu chỉnh sử dụng hiệu của toán tử $B$ tại hai điểm $x^k$ và $y^k$.

Do đó, thuật toán lặp được xác định bởi:

$$
\boxed{
\begin{aligned}
y^k&=J_{\gamma A}(x^k-\gamma Bx^k),\\
x^{k+1}&=y^k-\gamma(By^k-Bx^k),
\quad k\geq0.
\end{aligned}
}
$$

Hay dạng phương trình điểm bất động:

$$
\boxed{
x^{k+1}=T_{\mathrm{MFBSM}}(x^k),
}
$$

trong đó:

$$
T_{\mathrm{MFBSM}}
=
(I-\gamma B)\circ J_{\gamma A}\circ(I-\gamma B)
+\gamma B.
$$

Thuật toán MFBSM tách việc cập nhật tại mỗi bước lặp $k$ thành ba bước nối tiếp:

## 1. Bước Forward thứ nhất:

Từ điểm hiện tại $x^k$, thực hiện một bước cập nhật trực tiếp theo toán tử $B$:

$$
z^k=x^k-\gamma B(x^k).
$$

## 2. Bước Backward:

Từ điểm trung gian $z^k$, áp dụng toán tử giải của $A$ để thu được điểm $y^k$:

$$
y^k=J_{\gamma A}(z^k)
\iff
y^k=J_{\gamma A}\left(x^k-\gamma B(x^k)\right).
$$

Trong trường hợp $A=\partial g$, với $g$ là một hàm lồi chính thường, nửa liên tục dưới, resolvent của $A$ chính là toán tử proximal:

$$
J_{\gamma A}
=
J_{\gamma\partial g}
=
\operatorname{prox}_{\gamma g}.
$$

Khi đó bước Backward có thể viết dưới dạng:

$$
y^k=\operatorname{prox}_{\gamma g}(z^k).
$$

## 3. Bước Forward thứ hai (hiệu chỉnh):

Sau khi tính được điểm $y^k$, thuật toán thực hiện thêm một bước hiệu chỉnh dựa trên sự thay đổi của toán tử $B$:

$$
\boxed{
x^{k+1}
=
y^k-\gamma\left(B(y^k)-B(x^k)\right).
}
$$

Bước hiệu chỉnh này là điểm khác biệt cơ bản giữa MFBSM của Tseng và phương pháp FBM thông thường.

Nhờ bước hiệu chỉnh, MFBSM có thể được áp dụng cho lớp toán tử $B$ đơn điệu và liên tục Lipschitz mà không cần giả thiết cocoercivity.

Quá trình lặp được thực hiện cho đến khi sai khác giữa hai điểm liên tiếp đủ nhỏ với $\mathrm{tol}>0$ là ngưỡng sai số cho trước:

$$
\|x^{k+1}-x^k\|<\mathrm{tol},
$$

Ngoài ra, có thể sử dụng sai số tại bước Backward:

$$
\|x^k-y^k\|<\mathrm{tol}.
$$

Đặc biệt, nếu $x^k=y^k$ thì:

$$
x^k=J_{\gamma A}(x^k-\gamma Bx^k),
$$

suy ra:

$$
0\in Ax^k+Bx^k.
$$

Do đó $x^k$ là nghiệm của bài toán bao hàm biến phân.

### Mã giả Modified Forward–Backward Splitting Method (MFBSM_Tseng)

**Bước 0 (Khởi tạo).**

Chọn $x^0\in\mathcal H$, $\gamma\in(0,1/L)$, $\varepsilon>0$, số vòng lặp tối đa $K_{\max}$.

Đặt $k=0$.

**Bước 1.** Tính điểm trung gian:

$$
z^k=x^k-\gamma B(x^k).
$$

**Bước 2.** Tính:

$$
y^k=J_{\gamma A}(z^k).
$$

**Bước 3.** Thực hiện bước hiệu chỉnh:

$$
x^{k+1}
=
y^k-\gamma\left(B(y^k)-B(x^k)\right).
$$

**Bước 4.** Nếu

$$
\|x^{k+1}-x^k\|\leq\varepsilon
$$

hoặc $k+1=K_{\max}$, dừng thuật toán.

**Bước 5.** Đặt $k:=k+1$ và quay lại Bước 1.

### Tài liệu tham khảo

Tseng, P. (2000). *A modified forward–backward splitting method for maximal monotone mappings*. SIAM Journal on Control and Optimization, 38(2), 431–446.

Bauschke, H. H., & Combettes, P. L. (2017). *Convex Analysis and Monotone Operator Theory in Hilbert Spaces* (2nd ed.). Springer.

Cholamjiak, P., Hieu, D. V., & Cho, Y. J. (2021). *Relaxed Forward–Backward Splitting Methods for Solving Variational Inclusions and Applications*. Journal of Scientific Computing, 88, Article 85.